# 02 — Sleep EDF Dataset Building

This notebook builds a subject-level sleep feature dataset from prepared hypnogram data.

## Objectives

- load prepared hypnogram data;
- extract subject-level sleep features;
- combine extracted features into a structured dataset;
- save processed sleep features for downstream analysis.

In [1]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found. Make sure README.md and data/ exist.')

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

csv_files = [PROCESSED_DIR / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

PROJECT_ROOT = C:\Users\vi\Sleep-clean
RAW_DIR = C:\Users\vi\Sleep-clean\data\raw
PROCESSED_DIR = C:\Users\vi\Sleep-clean\data\processed
Found 1 CSV files
hypno_df.csv


In [2]:
INPUT_DIR = Path('hypnograms')
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [3]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

Found 0 CSV files


In [4]:
from pathlib import Path

INPUT_DIR = PROCESSED_DIR
INPUT_DIR.mkdir(exist_ok=True)

print("Created or already exists:", INPUT_DIR.resolve())

Created or already exists: C:\Users\vi\Sleep-clean\data\processed


In [5]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 7 CSV files
hypno_df.csv
mmash_modeling_dataset.csv
paired_results_df.csv
paired_summary.csv
processing_log.csv
sleep_features_all_subjects.csv
sleep_features_subject_001.csv


In [6]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
for p in Path('.').iterdir():
    print(p)

Current working directory:
C:\Users\vi\Sleep-clean\notebooks

Files and folders here:
.ipynb_checkpoints
01_sleep_edf_exploration.ipynb
02_sleep_edf_build_dataset.ipynb
03_sleep_edf_feature_engineering.ipynb
04_mmash_sleep_mental_health.ipynb
05_sleep_deprivation_cognition.ipynb


In [7]:
INPUT_DIR = PROCESSED_DIR
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [8]:
csv_files = [PROCESSED_DIR / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 1 CSV files
hypno_df.csv


In [9]:
def load_hypnogram_csv(file_path):
    """Load a hypnogram CSV file."""
    return pd.read_csv(file_path)


def parse_subject_id(file_path):
    """Extract subject ID from file name."""
    return Path(file_path).stem


def extract_sleep_features(hypno_df, subject_id):
    """Extract simple subject-level sleep-stage features from hypnogram data."""
    df = hypno_df.copy()

    possible_stage_columns = [
        "stage",
        "sleep_stage",
        "Sleep Stage",
        "description",
        "annotation",
    ]

    stage_col = None
    for col in possible_stage_columns:
        if col in df.columns:
            stage_col = col
            break

    if stage_col is None:
        stage_col = df.columns[-1]

    stage_counts = df[stage_col].value_counts(dropna=False)

    features = {
        "subject_id": subject_id,
        "n_epochs": len(df),
        "n_unique_stages": df[stage_col].nunique(dropna=True),
    }

    for stage, count in stage_counts.items():
        safe_stage = str(stage).replace(" ", "_").replace("/", "_")
        features[f"stage_{safe_stage}_count"] = count
        features[f"stage_{safe_stage}_ratio"] = count / len(df)

    return pd.DataFrame([features])

In [10]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

Testing file: hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,n_epochs,n_unique_stages,stage_Sleep_stage_3_count,stage_Sleep_stage_3_ratio,stage_Sleep_stage_2_count,stage_Sleep_stage_2_ratio,stage_Sleep_stage_1_count,stage_Sleep_stage_1_ratio,stage_Sleep_stage_4_count,stage_Sleep_stage_4_ratio,stage_Sleep_stage_W_count,stage_Sleep_stage_W_ratio,stage_Sleep_stage_R_count,stage_Sleep_stage_R_ratio
0,hypno_df,153,6,48,0.313725,40,0.261438,24,0.156863,23,0.150327,12,0.078431,6,0.039216


In [11]:
INPUT_DIR = PROCESSED_DIR
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [12]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

Found 7 CSV files
hypno_df.csv
mmash_modeling_dataset.csv
paired_results_df.csv
paired_summary.csv
processing_log.csv
sleep_features_all_subjects.csv
sleep_features_subject_001.csv


In [13]:
def clean_hypnogram(df):
    df = df.copy()
    df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()
    df = df[df['duration'] > 0].copy()
    df = df.reset_index(drop=True)
    return df

In [14]:
def find_sleep_boundaries(df):
    sleep_mask = df['description'] != 'Sleep stage W'

    if sleep_mask.sum() == 0:
        return None, None

    first_sleep_idx = df[sleep_mask].index[0]
    last_sleep_idx = df[sleep_mask].index[-1]

    return first_sleep_idx, last_sleep_idx

In [15]:
def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)
    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

In [16]:
def load_hypnogram_csv(file_path):
    df = pd.read_csv(file_path)

    if 'duration' not in df.columns and 'duration_sec' in df.columns:
        df = df.rename(columns={'duration_sec': 'duration'})

    required_cols = {'description', 'duration'}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    df = df[['description', 'duration']].copy()
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')

    return df

In [17]:
def parse_subject_id(file_path):
    return file_path.stem

In [18]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

Testing file: hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


In [19]:
from pathlib import Path

INPUT_DIR = PROCESSED_DIR
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [20]:
csv_files = [PROCESSED_DIR / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 1 CSV files
hypno_df.csv


In [21]:
if len(csv_files) == 0:
    print("No CSV files found. Re-run the path/file discovery cells.")
else:
    test_file = csv_files[0]
    print('Testing file:', test_file.name)

    test_hypno_df = load_hypnogram_csv(test_file)
    display(test_hypno_df.head())

    test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
    display(test_features)

Testing file: hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


In [22]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

In [23]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

sleep_features_df shape: (1, 13)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


processing_log_df shape: (1, 5)


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [24]:
processing_log_df['status'].value_counts(dropna=False)

status
success    1
Name: count, dtype: int64

In [25]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [26]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [27]:
from pathlib import Path

INPUT_DIR = PROCESSED_DIR
csv_files = [PROCESSED_DIR / 'hypno_df.csv']

print("csv_files:", csv_files)
print("len(csv_files):", len(csv_files))

csv_files: [WindowsPath('C:/Users/vi/Sleep-clean/data/processed/hypno_df.csv')]
len(csv_files): 1


In [28]:
test_file = csv_files[0]
print("Testing file:", test_file)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
display(test_features)

Testing file: C:\Users\vi\Sleep-clean\data\processed\hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


In [29]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

print("sleep_features_df shape:", sleep_features_df.shape)
print("processing_log_df shape:", processing_log_df.shape)

display(sleep_features_df)
display(processing_log_df)

sleep_features_df shape: (1, 13)
processing_log_df shape: (1, 5)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [30]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

sleep_features_df shape: (1, 13)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


processing_log_df shape: (1, 5)


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [31]:
processing_log_df['status'].value_counts(dropna=False)

status
success    1
Name: count, dtype: int64

In [32]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [33]:
sleep_features_df.to_csv(PROCESSED_DIR / 'sleep_features_all_subjects.csv', index=False)
processing_log_df.to_csv(PROCESSED_DIR / 'processing_log.csv', index=False)

In [34]:
check_df = pd.read_csv(PROCESSED_DIR / 'sleep_features_all_subjects.csv')
check_df

,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


## Conclusion

A structured subject-level sleep feature dataset was built from prepared hypnogram data.  
The resulting features are saved for further analysis in the feature engineering and mental health notebooks.